# Time is the Hardest Dimension

Time dimensions are special: they're naturally ordered, and they need an aggregation granularity (daily, weekly, monthly). This makes them the foundation for a family of transforms that are simple to express but non-trivial to implement in SQL.

This notebook demonstrates SLayer's time-related features with working code.

See also: [Formulas](../../concepts/formulas.md) | [Queries](../../concepts/queries.md)

**Prerequisites:** `pip install motley-slayer[examples]`

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, models = ensure_jaffle_shop()

/home/james/miniconda3/envs/motley3.11/lib/python3.11/site-packages/sqlglot/tokens.py:14: UserWarning: sqlglot[rs] is deprecated and no longer compatible with sqlglot. Please use sqlglotc instead for faster parsing: pip install sqlglot[c]
  warnings.warn(


## Time Dimensions and Granularity

A time dimension truncates a date/timestamp column to a given granularity. SLayer supports `day`, `week`, `month`, `quarter`, and `year`.

Let's start with monthly revenue:

In [2]:
result = engine.execute(query={
    "source_model": "orders",
    "time_dimensions": [{"dimension": {"name": "ordered_at"}, "granularity": "month"}],
    "fields": [{"formula": "order_total_sum"}],
    "order": [{"column": {"name": "ordered_at"}, "direction": "asc"}],
    "limit": 12,
})

print(f"{'Month':<12} {'Revenue':>14}")
print("-" * 28)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    print(f"{month:<12} ${rev:>13,.2f}")

Month               Revenue
----------------------------
2018-09      $     4,738.80
2018-10      $     7,882.42
2018-11      $    11,394.11
2018-12      $    14,904.61
2019-01      $    17,790.62
2019-02      $    17,464.28
2019-03      $    31,275.57
2019-04      $    36,448.96
2019-05      $    40,419.76
2019-06      $    34,389.80
2019-07      $    21,901.22
2019-08      $    24,826.53


## Month-over-Month Change

The `change()` transform computes the difference from the previous period. It needs no extra arguments — the time dimension and granularity are already known from the query. Internally it uses a self-join CTE (not LAG), so it handles time gaps correctly and never produces edge NULLs.

`change_pct()` gives the percentage change:

In [3]:
result = engine.execute(query={
    "source_model": "orders",
    "time_dimensions": [{"dimension": {"name": "ordered_at"}, "granularity": "month"}],
    "fields": [
        {"formula": "order_total_sum"},
        {"formula": "change(order_total_sum)", "name": "mom_change"},
        {"formula": "change_pct(order_total_sum)", "name": "mom_pct"},
    ],
    "order": [{"column": {"name": "ordered_at"}, "direction": "asc"}],
    "limit": 12,
})

print(f"{'Month':<12} {'Revenue':>14} {'MoM Change':>14} {'MoM %':>8}")
print("-" * 52)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    chg = row["orders.mom_change"]
    pct = row["orders.mom_pct"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    pct_str = f"{pct:+.1%}" if pct is not None else "-"
    print(f"{month:<12} ${rev:>13,.2f} {chg_str:>14} {pct_str:>8}")

Month               Revenue     MoM Change    MoM %
----------------------------------------------------
2018-09      $     4,738.80              -        -
2018-10      $     7,882.42        $+3,144   +66.3%
2018-11      $    11,394.11        $+3,512   +44.6%
2018-12      $    14,904.61        $+3,511   +30.8%
2019-01      $    17,790.62        $+2,886   +19.4%
2019-02      $    17,464.28          $-326    -1.8%
2019-03      $    31,275.57       $+13,811   +79.1%
2019-04      $    36,448.96        $+5,173   +16.5%
2019-05      $    40,419.76        $+3,971   +10.9%
2019-06      $    34,389.80        $-6,030   -14.9%
2019-07      $    21,901.22       $-12,489   -36.3%
2019-08      $    24,826.53        $+2,925   +13.4%


## Year-over-Year with time_shift

`time_shift(measure, offset, granularity)` shifts the time dimension by the given offset before applying the query's granularity. This is implemented as a calendar-based self-join CTE — semantically simple, but the SQL is non-trivial.

For example, `time_shift(order_total_sum, -1, 'year')` gives "same month, previous year":

In [4]:
result = engine.execute(query={
    "source_model": "orders",
    "time_dimensions": [{
        "dimension": {"name": "ordered_at"},
        "granularity": "month",
        "date_range": ["2020-01-01", "2020-12-31"],
    }],
    "fields": [
        {"formula": "order_total_sum"},
        {"formula": "time_shift(order_total_sum, -1, 'year')", "name": "prev_year"},
    ],
    "order": [{"column": {"name": "ordered_at"}, "direction": "asc"}],
})

print(f"{'Month':<12} {'Revenue':>14} {'Prev Year':>14}")
print("-" * 42)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    prev = row["orders.prev_year"]
    prev_str = f"${prev:>13,.2f}" if prev is not None else f"{'n/a':>14}"
    print(f"{month:<12} ${rev:>13,.2f} {prev_str}")

Month               Revenue      Prev Year
------------------------------------------
2020-01      $    73,865.47 $    17,790.62
2020-02      $    68,889.54 $    17,464.28
2020-03      $    73,909.92 $    31,275.57
2020-04      $    71,957.07 $    36,448.96
2020-05      $    97,235.94 $    40,419.76
2020-06      $    79,305.74 $    34,389.80
2020-07      $    46,089.70 $    21,901.22
2020-08      $    47,168.49 $    24,826.53
2020-09      $    77,802.64 $    36,567.21
2020-10      $   137,118.39 $    64,932.63
2020-11      $   139,967.48 $    65,816.05
2020-12      $   152,950.05 $    71,372.09


### time_shift vs lag

Without a granularity argument, `time_shift(x, -1)` is equivalent to `lag(x, 1)` — it uses SQL's `LAG()` window function to look at the previous row. This is more efficient but produces NULLs at the edges (the first row has no previous row).

With a granularity argument (like `'year'` above), `time_shift` uses a calendar-based self-join instead — no edge NULLs, and the shift can be any amount (not just a multiple of the query granularity).

## The last() Transform

`last(measure)` returns the most recent time period's value, broadcast to every row. This **collapses the time dimension** — useful for comparing each period to the latest one, or for filtering.

For example, revenue by store alongside each store's most recent month's revenue:

In [5]:
result = engine.execute(query={
    "source_model": "orders",
    "time_dimensions": [{"dimension": {"name": "ordered_at"}, "granularity": "quarter"}],
    "fields": [
        {"formula": "order_total_sum"},
        {"formula": "last(order_total_sum)", "name": "latest_quarter"},
    ],
    "dimensions": [{"name": "stores.name"}],
    "order": [{"column": {"name": "ordered_at"}, "direction": "desc"}],
    "limit": 10,
})

print(f"{'Quarter':<12} {'Store':<20} {'Revenue':>14} {'Latest Qtr':>14}")
print("-" * 62)
for row in result.data:
    qtr = str(row["orders.ordered_at"])[:7]
    store = row["orders.stores.name"]
    rev = row["orders.order_total_sum"]
    latest = row["orders.latest_quarter"]
    print(f"{qtr:<12} {store:<20} ${rev:>13,.2f} ${latest:>13,.2f}")

Quarter      Store                       Revenue     Latest Qtr
--------------------------------------------------------------
2021-07      New Orleans          $    16,026.36 $    25,132.84
2021-07      San Francisco        $    29,217.88 $    25,132.84
2021-07      Brooklyn             $    42,044.03 $    25,132.84
2021-07      Philadelphia         $    25,132.84 $    25,132.84
2021-07      Chicago              $    35,588.53 $    25,132.84
2021-04      Chicago              $   121,494.95 $    25,132.84
2021-04      New Orleans          $    31,543.17 $    25,132.84
2021-04      Brooklyn             $   131,430.94 $    25,132.84
2021-04      Philadelphia         $    77,151.29 $    25,132.84
2021-04      San Francisco        $   101,405.73 $    25,132.84


## Composing Transforms in Filters

Transforms can be composed and used in filters. For example, `last(change(order_total_sum)) < 0` keeps only rows where the most recent period's change was negative — i.e., revenue declined in the latest period.

This is a powerful pattern: "show me all stores, but only those whose most recent quarter-over-quarter revenue went down":

In [6]:
result = engine.execute(query={
    "source_model": "orders",
    "time_dimensions": [{"dimension": {"name": "ordered_at"}, "granularity": "quarter"}],
    "fields": [
        {"formula": "order_total_sum"},
        {"formula": "change(order_total_sum)", "name": "qoq_change"},
    ],
    "dimensions": [{"name": "stores.name"}],
    "filters": ["last(change(order_total_sum)) < 0"],
    "order": [{"column": {"name": "ordered_at"}, "direction": "desc"}],
    "limit": 10,
})

print("Stores where latest QoQ revenue declined:")
print(f"{'Quarter':<12} {'Store':<20} {'Revenue':>14} {'QoQ Change':>14}")
print("-" * 62)
for row in result.data:
    qtr = str(row["orders.ordered_at"])[:7]
    store = row["orders.stores.name"]
    rev = row["orders.order_total_sum"]
    chg = row["orders.qoq_change"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    print(f"{qtr:<12} {store:<20} ${rev:>13,.2f} {chg_str:>14}")

Stores where latest QoQ revenue declined:
Quarter      Store                       Revenue     QoQ Change
--------------------------------------------------------------


## Summary

Each of these concepts is simple and natural at the semantic level, but requires non-trivial SQL (CTEs, self-joins, window functions) to implement. SLayer offloads that effort.

| Transform | What it does | SQL implementation |
|-----------|-------------|-------------------|
| `change(x)` | Difference from previous period | Self-join CTE (gap-safe, no edge NULLs) |
| `change_pct(x)` | Percentage change from previous period | Same as change, divided |
| `time_shift(x, -1, 'year')` | Same measure, shifted in time | Calendar-based self-join CTE |
| `lag(x)` / `lead(x)` | Previous/next row's value | SQL LAG/LEAD window (NULLs at edges) |
| `last(x)` | Most recent period's value, broadcast | Sub-query with FIRST_VALUE |
| `cumsum(x)` | Running total over time | SUM window function |

All transforms compose: `cumsum(change(x))`, `last(change(x))` in filters, etc.

See the [Formulas docs](../../concepts/formulas.md) for the full reference.